# CENG493 — Turkish Legal RAG: Colab Setup

## Prerequisites — read before running anything

### 1. Set the runtime to T4 GPU
Go to **Runtime → Change runtime type → Hardware accelerator → T4 GPU**, then click **Save**.
The embedding and index-build steps are significantly faster on GPU. Forgetting this is the most common reason the notebook is slow.

### 2. Files required in your Google Drive
Place the following two CSV files inside a folder called `CENG493` in the root of your Google Drive (`My Drive/CENG493/`):

| File | Description |
|---|---|
| `combined_dataset.csv` | Main corpus of Turkish legal articles |
| `hmgs_2025_240_only_correct_answers_v2.csv` | HMGS benchmark QA pairs |

If your Drive folder path is different, update the `DRIVE_DATA_DIR` variable in **Cell 1** before running.

### 3. Notebook structure
| Cell | What it does | Required? |
|---|---|---|
| 1 | Mount Drive + copy datasets | Yes |
| 2 | Clone repo (`main` branch) | Yes |
| 3 | Install Python dependencies | Yes |
| 4 | Sanity-check config paths | Yes |
| 5 | Prepare data + build FAISS index | Yes |
| 6 | Retrieval evaluation (no LLM) | Yes |
| 7 | Ollama / QA generation note | — |
| 8 | Install and start Ollama | Optional |
| 9 | QA generation + QA evaluation | Optional (needs Cell 8) |

In [ ]:
# Cell 1 — Mount Google Drive and copy the required dataset files
#
# Adjust DRIVE_DATA_DIR if your CSVs live in a different Drive folder.
# The target paths below match what config.py expects.

from google.colab import drive
import shutil, pathlib

drive.mount('/content/drive')

DRIVE_DATA_DIR = pathlib.Path('/content/drive/MyDrive/CENG493')
DEST_DIR       = pathlib.Path('/content/CENG493_Project')
DEST_DIR.mkdir(parents=True, exist_ok=True)

files_needed = [
    'combined_dataset.csv',
    'hmgs_2025_240_only_correct_answers_v2.csv',
]

for fname in files_needed:
    src  = DRIVE_DATA_DIR / fname
    dest = DEST_DIR / fname
    if not src.exists():
        raise FileNotFoundError(
            f'Could not find {src}\n'
            f'Make sure the file is inside My Drive/CENG493/ (or update DRIVE_DATA_DIR above).'
        )
    shutil.copy(src, dest)
    print(f'Copied  {fname}  ->  {dest}')

print('\nDatasets ready.')

In [ ]:
# Cell 2 — Clone the repository (main branch)
!git clone --branch main --single-branch \
    https://github.com/BeratMert29/Turkish-Legal-Question-Answering-RAG.git \
    /content/CENG493_Project/CENG493_Project
%cd /content/CENG493_Project/CENG493_Project

In [ ]:
# Cell 3 — Install Python dependencies
# Colab already ships NumPy, pandas, and PyTorch; requirements-colab.txt
# pins only the additional packages this project needs.
!pip install -q -r requirements-colab.txt

In [ ]:
# Cell 4 — Sanity-check that config paths resolve to existing files
# Both lines should print True. If you see False, re-run Cell 1.
!python - <<'PY'
import config
print('RAW_DATA_PATH: ', config.RAW_DATA_PATH,  config.RAW_DATA_PATH.exists())
print('HMGS_DATA_PATH:', config.HMGS_DATA_PATH, config.HMGS_DATA_PATH.exists())
print('PROJECT_DIR:   ', config.BASE_DIR)
PY

In [ ]:
# Cell 5 — Prepare data and build the FAISS index
# 01_prepare_data.py chunks the legal corpus.
# 02_build_index.py encodes chunks and writes the FAISS index.
# Both scripts are idempotent; re-running them is safe.
!PYTHONUTF8=1 python scripts/01_prepare_data.py
!PYTHONUTF8=1 python scripts/02_build_index.py

In [ ]:
# Cell 6 — Retrieval evaluation (no LLM required)
# Evaluates dense retrieval quality against both benchmarks.
# This is the core experiment and does not require Ollama.
!PYTHONUTF8=1 python scripts/03_evaluate_retrieval.py --dataset hmgs
!PYTHONUTF8=1 python scripts/03_evaluate_retrieval.py --dataset qa300

## Optional: QA generation with a local LLM via Ollama

Scripts `04_generate_answers.py` and `05_evaluate_qa.py` call a locally running LLM through Ollama.
**You can skip Cells 8 and 9 if you only need retrieval metrics** (Cell 6 above).

**Free-tier T4 warning:** Ollama runs the model on CPU in Colab (the CUDA driver version Colab ships
is sometimes incompatible with the Ollama GPU backend). CPU inference for `qwen2.5:7b-instruct`
takes roughly 30–90 seconds per question, so generating answers for 240 HMGS questions can take
several hours on a free-tier session. Consider using a smaller model or a subset of the dataset
for a quick smoke test.

In [ ]:
# Cell 8 — Install and start Ollama, then pull the model  [OPTIONAL]
# Skip this cell (and Cell 9) if you only need retrieval evaluation.

!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
subprocess.Popen(['ollama', 'serve'])
time.sleep(5)   # give the server a moment to start

!ollama pull qwen2.5:7b-instruct

In [ ]:
# Cell 9 — QA generation and evaluation  [OPTIONAL — requires Cell 8]
# Generates model answers for the dense-retrieval pipeline and scores them.
!PYTHONUTF8=1 python scripts/04_generate_answers.py --mode dense
!PYTHONUTF8=1 python scripts/05_evaluate_qa.py      --mode dense